In [1]:
%%writefile dashboard_app.py
import pandas as pd
import plotly.express as px
import streamlit as st
import numpy as np

# --- 1. Data Loading and Preprocessing ---
# Read the data from the specified local file
df = pd.read_csv('/content/predictive_maintenance.csv', sep=',', on_bad_lines='skip')

# Dynamically find the failure column
failure_columns = [col for col in df.columns if 'failure' in col.lower()]

if not failure_columns:
    st.error("Could not find any 'failure' related columns in the data. Available columns: " + str(df.columns.tolist()))
    raise KeyError("No failure column found!")
elif len(failure_columns) > 1:
    if 'Machine failure' in failure_columns:
        failure_col = 'Machine failure'
    elif 'failure_type' in failure_columns:
        failure_col = 'failure_type'
    else:
        failure_col = failure_columns[0]
    st.warning(f"Multiple failure columns found: {failure_columns}. Using: {failure_col}")
else:
    failure_col = failure_columns[0]

# Create a binary 'is_failed' column for consistent failure indication
if df[failure_col].dtype == 'object':
    df['is_failed'] = (df[failure_col] != "No Failure").astype(int)
else:
    df['is_failed'] = df[failure_col]

# --- 2. Simulate XGBoost Model Outputs (based on user's provided values) ---

# Scale confusion matrix values to match DataFrame size (10000 rows)
tn_scaled = 1867 * (len(df) / 2000) # 9335
fp_scaled = 65 * (len(df) / 2000)   # 325
fn_scaled = 13 * (len(df) / 2000)   # 65
tp_scaled = 55 * (len(df) / 2000)   # 275

# Simulate predicted probabilities and status
p_failed_given_actual_failed = np.random.uniform(0.7, 0.95, int(tp_scaled)) # TP (actual failed, predicted failed)
p_failed_given_actual_healthy_as_fp = np.random.uniform(0.55, 0.8, int(fp_scaled)) # FP (actual healthy, predicted failed)
p_failed_given_actual_failed_as_fn = np.random.uniform(0.1, 0.45, int(fn_scaled)) # FN (actual failed, predicted healthy)
p_failed_given_actual_healthy = np.random.uniform(0.05, 0.3, int(tn_scaled)) # TN (actual healthy, predicted healthy)

# Combine and shuffle probabilities
predicted_proba = np.concatenate([
    p_failed_given_actual_failed,
    p_failed_given_actual_healthy_as_fp,
    p_failed_given_actual_failed_as_fn,
    p_failed_given_actual_healthy
])
np.random.shuffle(predicted_proba)

df['predicted_proba'] = predicted_proba[:len(df)] # Trim if necessary
df['predicted_status'] = (df['predicted_proba'] > 0.5).astype(int)
df['predicted_status_label'] = df['predicted_status'].map({0: 'Predicted Healthy', 1: 'Predicted Failure'})

# Hardcoded Metrics (from user's prompt)
accuracy = 96.1
precision = 45.8
recall = 80.9
f1_score = 58.5

# Hardcoded Feature Importance (from user's prompt)
feature_importance_data = {
    'Feature': ['Torque [Nm]', 'Rotational speed [rpm]', 'Tool wear [min]', 'Type', 'Air temperature [K]', 'Process temperature [K]'],
    'Importance': [0.40, 0.30, 0.15, 0.08, 0.04, 0.03]
}
feature_importance_df = pd.DataFrame(feature_importance_data).sort_values(by='Importance', ascending=True)

# Hardcoded Confusion Matrix (from user's prompt, scaled)
confusion_matrix_values = {
    'Predicted Healthy': [int(tn_scaled), int(fn_scaled)],
    'Predicted Failure': [int(fp_scaled), int(tp_scaled)]
}
confusion_matrix_df = pd.DataFrame(confusion_matrix_values, index=['Actual Healthy', 'Actual Failure'])


# --- 3. Streamlit Dashboard Layout ---
st.set_page_config(layout="wide")
st.title("Predictive Maintenance Dashboard: XGBoost Model Insights")

# --- Add header details ---
st.markdown("Dashboard by: **Ahmed Hussein**")
st.markdown("Initiative: **DEPI**")
st.markdown("Company: **YAT Learning Solution**")
st.markdown("---") # Add a separator for better visual structure

# --- Filters (Reusing from previous context where appropriate) ---
st.sidebar.header("Dashboard Filters")

machine_types = df['Type'].unique().tolist()
selected_machine_type = st.sidebar.multiselect(
    'Select Machine Type', options=machine_types, default=machine_types
)

failure_types = df[failure_col].unique().tolist()
selected_failure_type = st.sidebar.multiselect(
    'Select Actual Failure Type', options=failure_types, default=failure_types
)

predicted_statuses = df['predicted_status_label'].unique().tolist()
selected_predicted_status = st.sidebar.multiselect(
    'Select Predicted Status', options=predicted_statuses, default=predicted_statuses
)

# Apply filters
filtered_df = df[
    (df['Type'].isin(selected_machine_type)) &
    (df[failure_col].isin(selected_failure_type)) &
    (df['predicted_status_label'].isin(selected_predicted_status))
]

# --- 4. KPI Cards ---
st.header("Model Performance KPIs")
col1, col2, col3, col4 = st.columns(4)
col1.metric("Accuracy", f"{accuracy:.1f}%")
col2.metric("Precision", f"{precision:.1f}%")
col3.metric("Recall", f"{recall:.1f}%")
col4.metric("F1 Score", f"{f1_score:.1f}%")

# --- 5. Feature Importance ---
st.header("Feature Importance (XGBoost)")
fig_feature_importance = px.bar(
    feature_importance_df,
    x='Importance', y='Feature',
    orientation='h',
    title='Most Influential Features for Failure Prediction',
    height=400
)
fig_feature_importance.update_layout(showlegend=False, yaxis={'categoryorder':'total ascending'})
st.plotly_chart(fig_feature_importance, use_container_width=True)

# --- 6. Confusion Matrix ---
st.header("Confusion Matrix")
st.markdown("*(Values are scaled to match dataset size)*")

def color_confusion_matrix(val):
    # Apply color based on value, darker for higher values
    if isinstance(val, (int, float)):
        if val > 0:
            # Scale values to a 0-1 range for color mapping
            max_val = confusion_matrix_df.values.max()
            normalized_val = val / max_val
            # Darker color for higher values
            return f'background-color: rgba(0, 100, 0, {normalized_val * 0.7}); color: white'
    return ''

st.dataframe(confusion_matrix_df.style.applymap(color_confusion_matrix), use_container_width=True)


# --- 7. Failure Probability Distribution ---
st.header("Failure Probability Distribution")
fig_prob_dist = px.histogram(filtered_df, x='predicted_proba', nbins=50,
                             title='Distribution of Predicted Failure Probabilities',
                             labels={'predicted_proba': 'Predicted Probability of Failure'})
fig_prob_dist.update_layout(xaxis_title='Predicted Probability', yaxis_title='Number of Machines')
st.plotly_chart(fig_prob_dist, use_container_width=True)

# --- 8. High-Risk Machines Table ---
st.header("High-Risk Machines")
# Filter for predicted failures and sort by highest probability
high_risk_machines = filtered_df[filtered_df['predicted_status'] == 1].sort_values(by='predicted_proba', ascending=False)

if not high_risk_machines.empty:
    st.dataframe(high_risk_machines[['Product ID', 'Type', 'Torque [Nm]', 'Tool wear [min]', 'predicted_proba', 'predicted_status_label']], use_container_width=True)
else:
    st.info("No high-risk machines found based on current filters.")

# --- 9. Key Insights Panel ---
st.header("Key Insights")
st.markdown(
    """
    - The XGBoost model achieved **96.1% accuracy** while maintaining an **80.9% recall rate** for failure detection.
    - **Torque and rotational speed** are the most influential factors affecting machine failures.
    - The model successfully detects most failure events, missing only a small number of actual failures.
    - Most prediction errors are **false positives**, which is acceptable in predictive maintenance scenarios where avoiding missed failures is critical.
    - High-risk machines can be identified and prioritized for preventive maintenance, reducing downtime and operational costs.
    """
)

Writing dashboard_app.py


In [2]:
import os
import sys
import subprocess
import time

app_file = os.path.join(r'C:\Users\Al Jazeera\Downloads', 'dashboard_app.py')
if not os.path.exists(app_file):
    raise FileNotFoundError(f"Streamlit app not found: {app_file}")

print('Starting Streamlit app...')
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', app_file, '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if hasattr(subprocess, 'CREATE_NEW_PROCESS_GROUP') else 0,
    text=True
)

print('Waiting for the app to start...')
time.sleep(5)
print('Streamlit should be running at: http://localhost:8501')
print('If you need a public URL, install pyngrok or localtunnel separately.')

Starting Streamlit app...
Waiting for the app to start...
Streamlit should be running at: http://localhost:8501
If you need a public URL, install pyngrok or localtunnel separately.
